# Singular Value Decomposition (SVD)

---

## Overview

**SVD** factorizes any matrix $A \in \mathbb{R}^{m \times n}$ as:

$$A = U \Sigma V^\top$$

where:
- $U \in \mathbb{R}^{m \times m}$: left singular vectors (directions in sample space)
- $\Sigma \in \mathbb{R}^{m \times n}$: diagonal matrix of **singular values** $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$
- $V^\top \in \mathbb{R}^{n \times n}$: right singular vectors (directions in feature space)

**Truncated SVD** keeps only the top $k$ components, giving the best rank-$k$ approximation:

$$A_k = U_k \Sigma_k V_k^\top$$

**Explained variance ratio:** $\dfrac{\sigma_i^2}{\sum_j \sigma_j^2}$

**Relationship to PCA:** SVD on the centered data matrix $\tilde{X}$ gives the same principal components as eigendecomposition of the covariance matrix.

---

**Dataset:** Food Nutrition (or digits images)  
**Task:** Low-rank approximation and dimensionality reduction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import TruncatedSVD
from rice_ml.preprocess import StandardScaler

In [ ]:
try:
    df = pd.read_csv('../../../data/FOOD-DATA-GROUP1.csv')
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    X = df[num_cols].dropna().values.astype(float)
    print(f'Loaded food nutrition dataset (FOOD-DATA-GROUP1): {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_digits
    digits = load_digits()
    X = digits.data.astype(float)
    print(f'CSV not found. using digits dataset: {X.shape}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled shape: {X_scaled.shape}')

## Singular Values. How Much Information Do They Capture?

We plot singular values and their cumulative explained variance to choose $k$.

In [ ]:
_, s, _ = np.linalg.svd(X_scaled, full_matrices=False)
var_ratio = (s ** 2) / (s ** 2).sum()
cumulative = np.cumsum(var_ratio)

n_show = min(20, len(s))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.bar(range(1, n_show + 1), s[:n_show], color='steelblue')
ax1.set_xlabel('Component', fontsize=14)
ax1.set_ylabel('Singular Value $\sigma_i$', fontsize=14)
ax1.set_title('Singular Values', fontsize=16)

ax2.plot(range(1, n_show + 1), cumulative[:n_show] * 100, marker='o', color='salmon')
ax2.axhline(90, color='black', linestyle='--', label='90% threshold')
ax2.set_xlabel('Number of Components $k$', fontsize=14)
ax2.set_ylabel('Cumulative Variance (%)', fontsize=14)
ax2.set_title('Cumulative Explained Variance', fontsize=16)
ax2.legend(fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Project to 2D with TruncatedSVD
svd = TruncatedSVD(n_components=2)
X_proj = svd.fit_transform(X_scaled)

print(f'Explained variance ratio (k=2): {svd.explained_variance_ratio_.sum():.3f}')

plt.figure(figsize=(10, 8))
plt.scatter(X_proj[:, 0], X_proj[:, 1], alpha=0.5, color='steelblue')
plt.xlabel('SVD Component 1', fontsize=15)
plt.ylabel('SVD Component 2', fontsize=15)
plt.title('TruncatedSVD: 2D Projection', fontsize=18)
plt.show()

In [ ]:
# Low-rank reconstruction quality
k_values = [1, 2, 5, 10, 20]
errors = []

for k in k_values:
    if k >= min(X_scaled.shape):
        break
    svd_k = TruncatedSVD(n_components=k)
    X_reduced = svd_k.fit_transform(X_scaled)
    X_recon = svd_k.inverse_transform(X_reduced)
    errors.append(np.mean((X_scaled - X_recon) ** 2))

plt.figure(figsize=(10, 6))
plt.plot(k_values[:len(errors)], errors, marker='o', color='steelblue')
plt.xlabel('Number of Components $k$', fontsize=15)
plt.ylabel('Reconstruction MSE', fontsize=15)
plt.title('SVD: Reconstruction Error vs $k$', fontsize=18)
plt.show()

## Interpretation

- The **singular values** $\sigma_i$ decay rapidly. the first few components capture most of the information.
- SVD gives the **best rank-$k$ approximation** in the Frobenius norm sense (Eckart-Young theorem).
- Used in recommendation systems (collaborative filtering), image compression, and NLP (LSA).